# Phase 1 — Self-Training 파이프라인

**전제 조건 (실행 전 확인)**
```bash
venv/bin/python src/phase1_selftraining.py --step splits   # 1-1
venv/bin/python src/phase1_selftraining.py --step embed    # 1-2 (수 시간)
```

**이 노트북에서 하는 것**
- 1-3: MERT 임베딩 위에 Linear head 학습 → baseline F1 측정
- 1-4: Self-training 루프 (confidence threshold τ sweep)
- 1-5: 학습 곡선 + τ 트레이드오프 시각화

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay
from tqdm import tqdm   # tqdm.notebook 대신 plain tqdm 사용

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))

CLASSES = ['ordinario', 'pizzicato_secco', 'tremolo', 'sul_ponticello', 'col_legno_battuto']
N_CLASSES = len(CLASSES)
SEED_RATIOS = [0.05, 0.10, 0.20, 0.50]
TAU_LIST    = [0.7, 0.8, 0.9]
N_TRIALS    = 3
MAX_ITER    = 5

DEVICE = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'device: {DEVICE}')

In [11]:
EMB_PATH    = ROOT / 'data' / 'mert_embeddings.npy'
LAB_PATH    = ROOT / 'data' / 'mert_labels.npy'
META_PATH   = ROOT / 'data' / 'mert_meta.csv'
SPLITS_PATH = ROOT / 'data' / 'split_configs.json'

assert EMB_PATH.exists(),    f'임베딩 없음 — 먼저 --step embed 실행: {EMB_PATH}'
assert SPLITS_PATH.exists(), f'분할 설정 없음 — 먼저 --step splits 실행: {SPLITS_PATH}'

embeddings = np.load(EMB_PATH)          # (N, 768)
labels     = np.load(LAB_PATH)          # (N,)
df_meta    = pd.read_csv(META_PATH)

with open(SPLITS_PATH) as f:
    split_cfg = json.load(f)

print(f'임베딩 shape : {embeddings.shape}')
print(f'라벨 shape  : {labels.shape}')
print(f'split 수    : {len(split_cfg["splits"])}')

# 클래스 가중치 (불균형 대응)
counts = np.bincount(labels, minlength=N_CLASSES).astype(float)
class_weights = torch.tensor(
    (counts.sum() / (N_CLASSES * counts)), dtype=torch.float32
).to(DEVICE)
print('\n클래스 가중치:')
for cls, w in zip(CLASSES, class_weights.cpu().numpy()):
    print(f'  {cls:<22} {w:.3f}')

임베딩 shape : (3402, 768)
라벨 shape  : (3402,)
split 수    : 48

클래스 가중치:
  ordinario              0.570
  pizzicato_secco        1.065
  tremolo                0.654
  sul_ponticello         2.253
  col_legno_battuto      2.997


## 0단계 — CLAP 제로샷 기준선

"라벨이 전혀 없을 때" 성능을 먼저 측정한다.  
`laion/clap-htsat-unfused`로 텍스트 프롬프트 기반 제로샷 분류 — 학습 없음.  
이 수치가 학습 곡선의 하한 기준점이 된다.

In [12]:
from transformers import ClapModel, ClapProcessor

CLAP_MODEL_ID = "laion/clap-htsat-unfused"
CLAP_SR       = 48000

CLAP_PROMPTS = [
    "a string instrument playing with normal arco bowing",
    "a string instrument playing pizzicato, plucking the string",
    "a string instrument playing tremolo, rapid repeated bow strokes",
    "a string instrument playing sul ponticello, bowing near the bridge",
    "a string instrument playing col legno battuto, striking with the wood of the bow",
]

print(f"CLAP 모델 로드 중: {CLAP_MODEL_ID}")
clap_proc  = ClapProcessor.from_pretrained(CLAP_MODEL_ID)
clap_model = ClapModel.from_pretrained(CLAP_MODEL_ID)
clap_model.eval()
clap_model = clap_model.to(DEVICE)
print("로드 완료")

# transformers 5.x에서 get_text_features()가 tensor 대신 객체를 반환하는 문제 우회
# text_model → text_projection 직접 호출
with torch.no_grad():
    t_in       = clap_proc(text=CLAP_PROMPTS, return_tensors="pt", padding=True).to(DEVICE)
    t_out      = clap_model.text_model(**t_in)
    text_feats = clap_model.text_projection(t_out.pooler_output)   # (5, D)
    text_feats = text_feats / text_feats.norm(dim=-1, keepdim=True)

print(f"텍스트 임베딩 shape: {text_feats.shape}")

CLAP 모델 로드 중: laion/clap-htsat-unfused



Loading weights: 100%|██████████| 447/447 [00:00<00:00, 68900.59it/s]

로드 완료
텍스트 임베딩 shape: torch.Size([5, 512])


In [13]:
import librosa as _librosa

DATA_ROOT_CLAP   = ROOT / '_OrchideaSOL2020_release' / 'OrchideaSOL2020' / 'Strings'
INSTRUMENTS_CLAP = ['Violin', 'Viola', 'Violoncello', 'Contrabass']

# 오디오 파일 스캔
audio_files, audio_labels_clap = [], []
for inst in INSTRUMENTS_CLAP:
    for ci, cls in enumerate(CLASSES):
        folder = DATA_ROOT_CLAP / inst / cls
        if not folder.exists():
            continue
        for fp in sorted(folder.glob('*.wav')):
            audio_files.append(fp)
            audio_labels_clap.append(ci)

print(f"총 {len(audio_files)}개 파일 스캔")

# CLAP 추론 — audio_model + audio_projection 직접 호출
clap_preds = []
for fp in tqdm(audio_files, desc="CLAP 추론"):
    wav, _ = _librosa.load(str(fp), sr=CLAP_SR, mono=True)
    with torch.no_grad():
        a_in  = clap_proc(audio=wav, return_tensors="pt",       # audios → audio (transformers 5.x)
                          sampling_rate=CLAP_SR).to(DEVICE)
        a_out = clap_model.audio_model(**a_in)
        af    = clap_model.audio_projection(a_out.pooler_output)
        af    = af / af.norm(dim=-1, keepdim=True)
    sims = (af @ text_feats.T).squeeze(0)
    clap_preds.append(sims.argmax().item())

if DEVICE == 'mps':
    torch.mps.empty_cache()

clap_macro = f1_score(audio_labels_clap, clap_preds,
                      average='macro', zero_division=0)
clap_per   = f1_score(audio_labels_clap, clap_preds,
                      average=None, zero_division=0, labels=list(range(N_CLASSES)))

print(f"\nCLAP 제로샷 Macro F1 : {clap_macro:.4f}")
print(f"v1 SVM               : 0.9160")
for cls, val in zip(CLASSES, clap_per):
    print(f"  {cls:<22} {val:.4f}")

총 3402개 파일 스캔


ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# CSV 저장
pd.DataFrame({
    'file':       [f.name for f in audio_files],
    'true_label': [CLASSES[t] for t in audio_labels_clap],
    'pred_label': [CLASSES[p] for p in clap_preds],
}).to_csv(RESULTS_DIR / 'clap_zeroshot.csv', index=False)

# Confusion matrix
cm_clap = confusion_matrix(audio_labels_clap, clap_preds, labels=list(range(N_CLASSES)))
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(confusion_matrix=cm_clap, display_labels=CLASSES).plot(
    ax=ax, cmap='Blues', colorbar=False, xticks_rotation=45
)
ax.set_title(f'CLAP 제로샷 Confusion Matrix  (Macro F1={clap_macro:.3f})', fontsize=12)
plt.tight_layout()
fig.savefig(RESULTS_DIR / 'clap_confusion.png', dpi=150)
plt.show()
print(f'저장: {RESULTS_DIR}/clap_zeroshot.csv')
print(f'저장: {RESULTS_DIR}/clap_confusion.png')

## 1-3 — Classification Head + Baseline

In [ ]:
class LinearHead(nn.Module):
    """MERT 임베딩 (768) → 5 클래스. MERT 백본은 동결 상태이므로 이 head만 학습."""
    def __init__(self, in_dim: int = 768, n_classes: int = N_CLASSES, dropout: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Dropout(dropout),
            nn.Linear(in_dim, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [ ]:
def make_tensors(idx: list) -> tuple:
    """인덱스 목록 → (X, y) GPU tensor"""
    X = torch.tensor(embeddings[idx], dtype=torch.float32).to(DEVICE)
    y = torch.tensor(labels[idx],     dtype=torch.long).to(DEVICE)
    return X, y


def train_head(
    labeled_idx: list,
    val_idx:     list,
    epochs:      int   = 150,
    lr:          float = 1e-3,
    patience:    int   = 15,
) -> tuple:
    """
    labeled_idx로 LinearHead를 학습하고
    (model, best_val_f1, train_history) 반환.
    val_idx는 labeled_idx의 20%를 떼어낸 validation용.
    """
    model = LinearHead().to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    X_tr, y_tr = make_tensors(labeled_idx)
    X_val, y_val = make_tensors(val_idx)

    best_f1, best_state, wait = 0.0, None, 0
    history = []

    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()
        loss = criterion(model(X_tr), y_tr)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            preds = model(X_val).argmax(1).cpu().numpy()
        val_f1 = f1_score(y_val.cpu().numpy(), preds, average='macro', zero_division=0)
        history.append(val_f1)

        if val_f1 > best_f1:
            best_f1   = val_f1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_f1, history


def evaluate(model: nn.Module, test_idx: list) -> tuple:
    """(macro_f1, per_class_f1, conf_matrix) 반환"""
    model.eval()
    X_te, y_te = make_tensors(test_idx)
    with torch.no_grad():
        preds = model(X_te).argmax(1).cpu().numpy()
    y_true = y_te.cpu().numpy()
    macro  = f1_score(y_true, preds, average='macro',  zero_division=0)
    per_cls = f1_score(y_true, preds, average=None,    zero_division=0, labels=list(range(N_CLASSES)))
    cm      = confusion_matrix(y_true, preds, labels=list(range(N_CLASSES)))
    return macro, per_cls, cm


def split_val(labeled_idx: list, val_ratio: float = 0.2, seed: int = 0) -> tuple:
    """labeled_idx를 (train, val) 로 나눔 (stratified)"""
    rng = np.random.default_rng(seed)
    idx = np.array(labeled_idx)
    lab = labels[idx]
    tr, va = [], []
    for c in range(N_CLASSES):
        pos = idx[lab == c]
        n_val = max(1, round(len(pos) * val_ratio))
        chosen = rng.choice(len(pos), n_val, replace=False)
        va.extend(pos[chosen].tolist())
        tr.extend(np.delete(pos, chosen).tolist())
    return tr, va

In [ ]:
# ── Baseline 실행 (self-training 없이 지도학습만) ─────────────────────────────
# fold 0 (Viola 제외) 만 먼저 실행. 전체 fold는 아래 full run 셀 사용.

baseline_results = []   # {fold, ratio, trial, macro_f1, per_class_f1}

fold0_splits = [s for s in split_cfg['splits'] if s['fold'] == 0]

for s in tqdm(fold0_splits, desc='Baseline (fold=0)'):
    tr_idx, val_idx = split_val(s['labeled_idx'], seed=s['trial'])
    model, _, _ = train_head(tr_idx, val_idx)
    macro, per_cls, _ = evaluate(model, s['test_idx'])
    baseline_results.append({
        'fold':       s['fold'],
        'ratio':      s['seed_ratio'],
        'trial':      s['trial'],
        'macro_f1':   round(macro, 4),
        **{f'f1_{cls}': round(v, 4) for cls, v in zip(CLASSES, per_cls)},
    })

df_base = pd.DataFrame(baseline_results)
print('\nBaseline (fold=0, mean over trials):')
print(df_base.groupby('ratio')['macro_f1'].agg(['mean','std']).round(4))

## 1-4 — Self-Training 루프

In [ ]:
RESULTS_DIR = ROOT / 'results' / 'phase1'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def self_training(
    split:    dict,
    tau:      float,
    max_iter: int = MAX_ITER,
    verbose:  bool = False,
) -> dict:
    """
    단일 split + τ 에 대해 self-training을 실행한다.

    누수 방지 (결함 A 수정):
      - 원본 seed 라벨에서 iteration-선택 전용 val(`val_select`)을 1회 분리·고정.
      - val_select는 pseudo-labeling(unlabeled pool)·head early-stop·test 어디에도 쓰지 않음.
      - best iteration은 val_select F1로 선택, 그 iteration의 test F1을 보고.
        (기존: max(test F1) — test set에 iteration을 튜닝하는 누수였음)

    반환값 dict 키:
        iter_f1          : iteration별 TEST macro F1 리스트 (보고용)
        val_sel_f1       : iteration별 VAL_SELECT macro F1 리스트 (선택용)
        sel_iter         : val_select F1이 최대인 iteration 인덱스
        iter_acceptance  : iteration별 채택률 리스트
        iter_pseudo_acc  : iteration별 pseudo-label 정확도 (oracle 기준)
        iter_class_accept: iteration별 클래스별 채택 수
        final_cm         : 최종 confusion matrix
    """
    # ── iteration-선택 전용 val 분리 (원본 seed에서만, 한 번, 고정) ──────────────
    #    seed_core : head 학습 입력 풀의 시작점 (여기서 다시 early-stop val을 뗌)
    #    val_select: iteration 선택 전용 — 학습/pseudo/test와 완전 분리 (진짜 라벨)
    seed_core, val_select = split_val(
        split['labeled_idx'], val_ratio=0.2, seed=split.get('trial', 0) + 1000
    )

    labeled_pool  = list(seed_core)                    # 원본 seed - val_select
    unlabeled_idx = np.array(split['unlabeled_idx'])   # val_select와 원천적으로 분리됨
    test_idx      = split['test_idx']

    iter_f1, val_sel_f1 = [], []
    iter_acc, iter_pseudo_acc, iter_class_accept = [], [], []

    for it in range(max_iter):
        # 헤드 학습 (early-stop용 val은 labeled_pool 내부에서 별도 분리 — val_select와 무관)
        tr_idx, es_val_idx = split_val(labeled_pool, seed=it)
        model, _, _ = train_head(tr_idx, es_val_idx)

        # 평가: test(보고용) + val_select(선택용)
        macro, _, cm = evaluate(model, test_idx)
        iter_f1.append(round(macro, 4))
        vsel, _, _ = evaluate(model, val_select)
        val_sel_f1.append(round(vsel, 4))

        # 잔여 unlabeled에 대해 예측
        if len(unlabeled_idx) == 0:
            iter_acc.append(0.0)
            iter_pseudo_acc.append(None)
            iter_class_accept.append({c: 0 for c in CLASSES})
            if verbose:
                print(f'  iter {it}: unlabeled 소진')
            break

        model.eval()
        X_ul = torch.tensor(embeddings[unlabeled_idx], dtype=torch.float32).to(DEVICE)
        with torch.no_grad():
            probs = torch.softmax(model(X_ul), dim=1).cpu().numpy()  # (M, 5)

        max_prob  = probs.max(axis=1)          # (M,)
        pseudo_lbl = probs.argmax(axis=1)      # (M,)

        # confidence 필터
        accept_mask = max_prob >= tau
        n_accept    = accept_mask.sum()
        accept_rate = n_accept / len(unlabeled_idx)
        iter_acc.append(round(float(accept_rate), 4))

        # 클래스별 채택 수
        cls_accept = {}
        for ci, cls in enumerate(CLASSES):
            cls_accept[cls] = int(((pseudo_lbl == ci) & accept_mask).sum())
        iter_class_accept.append(cls_accept)

        # pseudo-label 정확도 (oracle)
        if n_accept > 0:
            oracle = labels[unlabeled_idx[accept_mask]]
            pseudo_acc = (pseudo_lbl[accept_mask] == oracle).mean()
            iter_pseudo_acc.append(round(float(pseudo_acc), 4))
        else:
            iter_pseudo_acc.append(None)

        if verbose:
            print(f'  iter {it}: test F1={macro:.4f} | val_sel F1={vsel:.4f} '
                  f'| 채택={n_accept} ({accept_rate:.1%}) | pseudo정확도={iter_pseudo_acc[-1]}')

        # 채택된 샘플을 labeled pool에 추가, unlabeled에서 제거
        accepted_global = unlabeled_idx[accept_mask]
        labeled_pool    = labeled_pool + accepted_global.tolist()
        unlabeled_idx   = unlabeled_idx[~accept_mask]

        # 조기 종료: 채택률 < 1%
        if accept_rate < 0.01:
            if verbose:
                print(f'  조기 종료: 채택률 {accept_rate:.2%} < 1%')
            break

    sel_iter = int(np.argmax(val_sel_f1)) if val_sel_f1 else -1

    return {
        'iter_f1':           iter_f1,
        'val_sel_f1':        val_sel_f1,
        'sel_iter':          sel_iter,
        'iter_acceptance':   iter_acc,
        'iter_pseudo_acc':   iter_pseudo_acc,
        'iter_class_accept': iter_class_accept,
        'final_cm':          cm,
        'n_iters':           len(iter_f1),
    }

In [ ]:
# ── Self-Training 실행 (fold=0, τ=0.8, 빠른 확인용) ─────────────────────────
# 전체 실행은 아래 full run 셀 사용

st_results = []  # {fold, ratio, trial, tau, baseline_f1, st_result}

TAU_QUICK  = 0.8   # 빠른 확인용 단일 τ

for s in tqdm(fold0_splits, desc=f'Self-Training fold=0 τ={TAU_QUICK}'):
    # baseline F1 (위에서 계산한 것 재사용)
    base_row  = df_base[(df_base['fold'] == s['fold']) &
                        (df_base['ratio'] == s['seed_ratio']) &
                        (df_base['trial'] == s['trial'])].iloc[0]
    base_f1   = base_row['macro_f1']

    result = self_training(s, tau=TAU_QUICK, verbose=False)

    # 누수 수정: iteration은 val_select F1로 선택, 그 iteration의 test F1을 보고
    sel = result['sel_iter']
    st_results.append({
        'fold':        s['fold'],
        'ratio':       s['seed_ratio'],
        'trial':       s['trial'],
        'tau':         TAU_QUICK,
        'baseline_f1': base_f1,
        'best_st_f1':  result['iter_f1'][sel] if result['iter_f1'] else base_f1,
        'sel_iter':    sel,
        'n_iters':     result['n_iters'],
        **{f'iter{i}_f1': result['iter_f1'][i]
           for i in range(len(result['iter_f1']))},
    })

df_st = pd.DataFrame(st_results)
print('\nSelf-Training vs Baseline (fold=0, τ=0.8, mean over trials):')
print(df_st.groupby('ratio')[['baseline_f1','best_st_f1']].mean().round(4))

### 전체 실행 (4 folds × 4 ratios × 3 trials × 3 τ)

시간이 오래 걸리므로 별도 셀로 분리. 위 quick run으로 방향 확인 후 실행.

In [ ]:
# ── 전체 실행 ─────────────────────────────────────────────────────────────────
# 시간 예상: fold 4 × ratio 4 × trial 3 × tau 3 × (학습+ST) ≈ 수 시간

all_results = []

total = len(split_cfg['splits']) * len(TAU_LIST)
pbar  = tqdm(total=total, desc='전체 실행')

for s in split_cfg['splits']:
    # baseline
    tr_idx, val_idx = split_val(s['labeled_idx'], seed=s['trial'])
    model, _, _ = train_head(tr_idx, val_idx)
    base_macro, base_per, _ = evaluate(model, s['test_idx'])

    for tau in TAU_LIST:
        result = self_training(s, tau=tau)

        # 누수 수정: iteration은 val_select F1로 선택, 그 iteration의 test F1을 보고
        sel = result['sel_iter']
        all_results.append({
            'fold':           s['fold'],
            'test_instrument': s['test_instrument'],
            'ratio':          s['seed_ratio'],
            'trial':          s['trial'],
            'tau':            tau,
            'baseline_f1':    round(base_macro, 4),
            'best_st_f1':     result['iter_f1'][sel] if result['iter_f1'] else round(base_macro, 4),
            'sel_iter':       sel,
            'n_iters':        result['n_iters'],
            'final_acceptance': result['iter_acceptance'][-1] if result['iter_acceptance'] else 0,
            # 재현성·검증용: 매 iteration의 test/val_select F1 배열 (JSON 문자열)
            'iter_f1_json':     json.dumps(result['iter_f1']),
            'val_sel_f1_json':  json.dumps(result['val_sel_f1']),
            **{f'base_f1_{cls}': round(v, 4) for cls, v in zip(CLASSES, base_per)},
        })
        pbar.update(1)

pbar.close()
df_all = pd.DataFrame(all_results)
df_all.to_csv(RESULTS_DIR / 'all_results.csv', index=False)
print(f'결과 저장: {RESULTS_DIR}/all_results.csv  ({len(df_all)}행)')

## 1-5 — 결과 시각화

In [ ]:
df_plot = df_all if 'df_all' in dir() else df_st

fig, ax = plt.subplots(figsize=(8, 5))

base_curve = df_plot.groupby('ratio')['baseline_f1'].agg(['mean','std'])
ax.plot(base_curve.index, base_curve['mean'], 'o--', color='gray',
        label='Baseline (supervised only)', linewidth=1.5)
ax.fill_between(base_curve.index,
                base_curve['mean'] - base_curve['std'],
                base_curve['mean'] + base_curve['std'], alpha=0.15, color='gray')

colors = {0.7: '#e07b54', 0.8: '#4c8be0', 0.9: '#54a87b'}
for tau in df_plot['tau'].unique():
    sub   = df_plot[df_plot['tau'] == tau]
    curve = sub.groupby('ratio')['best_st_f1'].agg(['mean','std'])
    ax.plot(curve.index, curve['mean'], 'o-', color=colors.get(tau, 'black'),
            label=f'Self-training τ={tau}', linewidth=1.5)
    ax.fill_between(curve.index,
                    curve['mean'] - curve['std'],
                    curve['mean'] + curve['std'], alpha=0.15, color=colors.get(tau, 'black'))

if 'clap_macro' in dir():
    ax.axhline(clap_macro, color='purple', linestyle=':', linewidth=1.2,
               label=f'CLAP zero-shot ({clap_macro:.3f})')
ax.axhline(0.916, color='red', linestyle=':', linewidth=1.2, label='v1 SVM (0.916)')

ax.set_xlabel('Labeled seed ratio', fontsize=12)
ax.set_ylabel('Macro F1', fontsize=12)
ax.set_title('Learning Curve: Label Ratio vs Performance', fontsize=13)
ax.set_xticks(SEED_RATIOS)
ax.set_xticklabels([f'{int(r*100)}%' for r in SEED_RATIOS])
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(RESULTS_DIR / 'learning_curve.png', dpi=150)
plt.show()
print('saved:', RESULTS_DIR / 'learning_curve.png')

In [ ]:
if 'df_all' in dir():
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    sub = df_all[df_all['ratio'] == 0.20]
    for fold_id in sub['fold'].unique():
        fd   = sub[sub['fold'] == fold_id]
        inst = fd['test_instrument'].iloc[0]
        axes[0].plot(fd.groupby('tau')['best_st_f1'].mean().index,
                     fd.groupby('tau')['best_st_f1'].mean().values, 'o-', label=f'test={inst}')
        axes[1].plot(fd.groupby('tau')['final_acceptance'].mean().index,
                     fd.groupby('tau')['final_acceptance'].mean().values, 'o-', label=f'test={inst}')
    axes[0].set_title('τ vs Macro F1 (seed=20%)', fontsize=12)
    axes[0].set_xlabel('τ (confidence threshold)'); axes[0].set_ylabel('Best Macro F1')
    axes[0].set_xticks(TAU_LIST); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)
    axes[1].set_title('τ vs Acceptance Rate (seed=20%)', fontsize=12)
    axes[1].set_xlabel('τ (confidence threshold)'); axes[1].set_ylabel('Acceptance rate')
    axes[1].set_xticks(TAU_LIST); axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)
    plt.tight_layout()
    fig.savefig(RESULTS_DIR / 'tau_tradeoff.png', dpi=150)
    plt.show()
else:
    print('Run full experiment (df_all) first.')

In [ ]:
if 'df_all' in dir():
    per_class_cols = [f'base_f1_{cls}' for cls in CLASSES]
    per_class_mean = df_all[df_all['tau'] == 0.8].groupby('ratio')[per_class_cols].mean()
    per_class_mean.columns = CLASSES

    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(SEED_RATIOS)); width = 0.15
    colors_cls = ['#4c8be0','#e07b54','#54a87b','#c06bbd','#c09e4a']
    for i, (cls, col) in enumerate(zip(CLASSES, colors_cls)):
        ax.bar(x + i*width, per_class_mean[cls], width, label=cls, color=col, alpha=0.85)
    ax.set_xticks(x + width*2)
    ax.set_xticklabels([f'{int(r*100)}%' for r in SEED_RATIOS])
    ax.set_xlabel('Labeled seed ratio'); ax.set_ylabel('Per-class F1')
    ax.set_title('Per-class F1 by Label Ratio (baseline)', fontsize=12)
    ax.legend(fontsize=8, loc='lower right'); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    fig.savefig(RESULTS_DIR / 'per_class_f1.png', dpi=150)
    plt.show()

    print('\n=== Phase 3 case study candidates ===')
    print('(classes ranked by mean F1 across all seed ratios)')
    for cls, f1 in per_class_mean.mean().sort_values().items():
        print(f'  {cls:<22} mean F1={f1:.4f}')
else:
    print('Run full experiment (df_all) first.')

In [ ]:
if 'df_all' in dir():
    best   = df_all.groupby(['ratio','tau'])[['baseline_f1','best_st_f1']].mean().round(4)
    v1_svm = 0.916

    # 결함 C 수정: 단일 fold(idxmax) 헤드라인 제거 → 4-fold 평균±표준편차로 통일
    # fold별로 trial 평균을 먼저 낸 뒤(4 fold), 그 4개로 mean±std 계산 (honest fold-level)
    lines_findings = []
    for ratio in sorted(df_all['ratio'].unique()):
        best_tau_r, best_m, best_s = None, -1.0, 0.0
        for tau in sorted(df_all['tau'].unique()):
            fold_means = (df_all[(df_all['ratio'] == ratio) & (df_all['tau'] == tau)]
                          .groupby('fold')['best_st_f1'].mean())
            if fold_means.mean() > best_m:
                best_m, best_s, best_tau_r = fold_means.mean(), fold_means.std(), tau
        lines_findings.append(
            f'- seed {int(ratio*100)}%: best τ={best_tau_r}, '
            f'self-training F1={best_m:.4f}±{best_s:.4f} (fold-level, n=4)\n'
        )

    fm50 = df_all[df_all['ratio'] == 0.5].groupby('fold')['baseline_f1'].mean()
    mert_full, mert_full_std = fm50.mean(), fm50.std()

    clap_line = f'CLAP zero-shot baseline : {clap_macro:.4f}\n' if 'clap_macro' in dir() else ''
    lines = [
        '# Phase 1 Results Summary\n\n',
        clap_line,
        f'v1 SVM baseline         : {v1_svm}\n\n',
        '> 누수 수정: iteration은 val_select F1로 선택(test 미사용). '
        '전 수치 4-fold 평균±표준편차, 단일 fold 헤드라인 제거.\n\n',
        '## Baseline vs Self-Training (mean over folds x trials)\n\n',
        best.to_markdown(), '\n\n',
        '## Key Findings (fold-level 평균±표준편차, n=4)\n\n',
        *lines_findings,
        f'- MERT head (seed=50%): F1={mert_full:.4f}±{mert_full_std:.4f} '
        f'({"exceeds" if mert_full > v1_svm else "below"} v1 SVM {v1_svm})\n',
        '- Self-training vs baseline 향상폭의 통계적 유의성은 '
        '`phase1_significance.py` 참조 (fold n=4, 검정력 한계 있음)\n',
    ]
    report_path = RESULTS_DIR / 'report.md'
    report_path.write_text(''.join(lines), encoding='utf-8')
    print(f'saved: {report_path}')
else:
    print('Run full experiment (df_all) first.')